In [ ]:
import json
import os
import chromadb
from embedder_api import embed_texts  # A2 endpoint — bge-base-en-v1.5


In [2]:
# Paths
CHUNKS_PATH   = "data/chunks/all_chunks.json"
VECTORSTORE   = "data/vectorstore"          # ChromaDB writes to disk here
COLLECTION    = "pet_care"                  # name of the collection inside ChromaDB


In [8]:
# ── Load chunks ───────────────────────────────────────────────────────────────
# Chunks are the pre-processed text segments produced by the chunker notebook.
# Each chunk is a dictionary with an id, text, and article metadata.
with open(CHUNKS_PATH, encoding="utf-8") as f:
    all_chunks = json.load(f)

# ⚠️ GUARD: Detect and remove duplicate chunk_ids before sending anything
# to ChromaDB. ChromaDB will reject the entire batch if even one duplicate
# exists — so we catch it here first and tell you clearly what went wrong.
seen_ids = set()
deduplicated_chunks = []
duplicate_ids = []

for chunk in all_chunks:
    chunk_id = chunk["chunk_id"]
    if chunk_id in seen_ids:
        duplicate_ids.append(chunk_id)
    else:
        seen_ids.add(chunk_id)
        deduplicated_chunks.append(chunk)

if duplicate_ids:
    print(f"⚠️  WARNING: Found {len(duplicate_ids)} duplicate chunk_ids in all_chunks.json.")
    print("   Duplicates were skipped. Check your chunker — the same article")
    print("   may have been chunked more than once.")
    print(f"   First 5 duplicates: {duplicate_ids[:5]}")

all_chunks = deduplicated_chunks
print(f"Loaded {len(all_chunks)} unique chunks")

Loaded 1597 unique chunks


In [9]:
# ── Initialize ChromaDB ───────────────────────────────────────────────────────
# ChromaDB is a database that stores vectors. "PersistentClient" means it
# saves to disk so we don't lose our work between sessions.
client = chromadb.PersistentClient(path=VECTORSTORE)

# get_or_create_collection: open the collection if it already exists,
# or create a fresh one if this is the first run.
# cosine similarity: the standard way to measure how "close" two vectors
# are for text search — angle between vectors, not raw distance.
collection = client.get_or_create_collection(
    name=COLLECTION,
    metadata={"hnsw:space": "cosine"},
)

print(f"ChromaDB collection '{COLLECTION}' ready")
print(f"Documents already in collection: {collection.count()}")


# ── Optional: reset the collection ───────────────────────────────────────────
# Uncomment this block if you want to wipe the collection and start fresh.
# WARNING: this deletes all stored embeddings. Only use it intentionally.
#
# client.delete_collection(COLLECTION)
# collection = client.get_or_create_collection(
#     name=COLLECTION,
#     metadata={"hnsw:space": "cosine"},
# )
# print("Collection reset.")


ChromaDB collection 'pet_care' ready
Documents already in collection: 0


In [10]:
# ── Embed and store ───────────────────────────────────────────────────────────
# Check which chunk_ids are already stored in ChromaDB.
# We fetch only the IDs (not the text or embeddings) to keep this fast.
existing_ids = set(collection.get(include=[])["ids"])

# Filter to only chunks that haven't been embedded yet.
to_embed = [c for c in all_chunks if c["chunk_id"] not in existing_ids]

print(f"Skipping {len(all_chunks) - len(to_embed)} already-embedded chunks")
print(f"Embedding {len(to_embed)} new chunks via A2 endpoint (bge-base-en-v1.5)...")

# 64 is a safe batch size for the A2 endpoint.
# Lower to 32 if you see timeout errors.
BATCH_SIZE = 64

for batch_start in range(0, len(to_embed), BATCH_SIZE):
    batch = to_embed[batch_start : batch_start + BATCH_SIZE]

    # Pull the fields ChromaDB needs from each chunk in this batch.
    ids       = [c["chunk_id"] for c in batch]
    texts     = [c["text"]     for c in batch]
    metadatas = [
        {
            "title":   c["title"],
            "url":     c["url"],
            "source":  c["source"],
            "species": c["species"],
            "topic":   c["topic"],
        }
        for c in batch
    ]

    # Call the A2 endpoint to convert texts into vectors.
    # embed_texts() handles the HTTP call and returns a list of 768-float vectors.
    # MUST use the same model here as in retriever.py — mixing models breaks retrieval.
    embeddings = embed_texts(texts)

    # Store vectors + original text + metadata in ChromaDB.
    # ChromaDB links them by position — ids[0] maps to embeddings[0], etc.
    collection.add(
        ids        = ids,
        embeddings = embeddings,
        documents  = texts,
        metadatas  = metadatas,
    )

    batch_end = min(batch_start + BATCH_SIZE, len(to_embed))
    print(f"  Embedded chunks {batch_start + 1}\u2013{batch_end} of {len(to_embed)}")

print(f"\nDone. Collection now contains {collection.count()} documents.")
print("Embedding model: bge-base-en-v1.5 (768-dim, A2 endpoint)")
